# PCA sensitivity analysis for descriptor-correlation thresholds

This notebook repeats the DBD, full-length (FL), and non-DBD amino-acid-composition PCA at `|r| > 0.50, 0.60, 0.70, 0.80`.

It preserves the primary workflow:

- use only columns ending in `_MolePct`;
- do not apply CLR transformation;
- connect descriptor pairs when `|r|` is **greater than** the threshold;
- identify connected components of the correlation graph;
- retain the descriptor with the largest variance from each component;
- apply `StandardScaler` before PCA.

The `0.60` result is the reference. Similarity of each PC1-PC2 configuration to the reference is evaluated using (i) Spearman correlation of pairwise sample distances and (ii) Procrustes disparity. Higher distance correlation and lower Procrustes disparity indicate greater similarity.


In [ ]:
# Upload all three input CSV files together and save them under /content.
from pathlib import Path
from google.colab import files

CONTENT_DIR = Path('/content')
REQUIRED_UPLOADS = {
    'aaComp_MolePct.csv',
    'FL_aaComp_MolePct.csv',
    'non_DBD_aaComp_MolePct.csv',
}

print('Select the following three files together:')
for name in sorted(REQUIRED_UPLOADS):
    print(' ', name)

uploaded = files.upload()
uploaded_names = set(uploaded)
missing = REQUIRED_UPLOADS - uploaded_names
if missing:
    raise ValueError(
        'Required files were not uploaded: ' + ', '.join(sorted(missing))
    )

# Save explicitly to the absolute paths used by the following cells.
for expected_name in sorted(REQUIRED_UPLOADS):
    destination = CONTENT_DIR / expected_name
    destination.write_bytes(uploaded[expected_name])
    print(f'Saved: {destination} ({destination.stat().st_size:,} bytes)')

extra = uploaded_names - REQUIRED_UPLOADS
if extra:
    print('Additional uploaded files were not used:', sorted(extra))

print('All three required CSV files are ready in /content.')


In [ ]:
import os
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import pdist
from scipy.spatial import procrustes
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path('/content')
OUTPUT_DIR = BASE_DIR / 'PCA_threshold_sensitivity'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = [0.50, 0.60, 0.70, 0.80]
REFERENCE_THRESHOLD = 0.60
N_COMPONENTS = 10

DATASETS = {
    'DBD': {
        'path': BASE_DIR / 'aaComp_MolePct.csv',
        'id_mode': 'SM00066',
    },
    'FL': {
        'path': BASE_DIR / 'FL_aaComp_MolePct.csv',
        'id_mode': 'FL',
    },
    'non_DBD': {
        'path': BASE_DIR / 'non_DBD_aaComp_MolePct.csv',
        'id_mode': 'non_DBD',
    },
}

for name, cfg in DATASETS.items():
    if not cfg['path'].exists():
        raise FileNotFoundError(f'Missing input for {name}: {cfg["path"]}')

print('Output directory:', OUTPUT_DIR)


In [ ]:
def clean_id_series(id_series, mode):
    """Apply the same ID cleaning used in the primary PCA notebook."""
    s = id_series.astype(str)
    base = s.str.split('|', n=1, expand=True)[0]
    base = base.str.replace('-P', '', regex=False)

    if mode == 'SM00066':
        counts = base.groupby(base).transform('count')
        dup_index = base.groupby(base).cumcount()
        base = np.where(
            counts > 1,
            base + '_' + (dup_index + 1).astype(str),
            base,
        )
        return pd.Series(base, index=id_series.index)
    if mode in {'FL', 'non_DBD'}:
        return base
    raise ValueError("mode must be 'SM00066', 'FL', or 'non_DBD'")


def select_descriptors_by_correlation(X, threshold):
    """Select one maximum-variance representative per correlation component."""
    corr = X.corr()
    abs_corr = corr.abs()
    columns = list(X.columns)
    adjacency = {column: set() for column in columns}

    for i, left in enumerate(columns):
        for j in range(i + 1, len(columns)):
            right = columns[j]
            if abs_corr.iloc[i, j] > threshold:
                adjacency[left].add(right)
                adjacency[right].add(left)

    components = []
    visited = set()
    for column in columns:
        if column in visited:
            continue
        stack = [column]
        visited.add(column)
        component = []
        while stack:
            current = stack.pop()
            component.append(current)
            for neighbor in adjacency[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)
        components.append(sorted(component))

    variances = X.var(axis=0)
    retained = []
    records = []
    for cluster_id, component in enumerate(components, start=1):
        representative = variances.loc[component].idxmax()
        retained.append(representative)
        for feature in component:
            records.append({
                'feature': feature,
                'cluster_id': cluster_id,
                'cluster_size': len(component),
                'variance': float(variances.loc[feature]),
                'representative': representative,
                'status': 'retained' if feature == representative else 'excluded',
                'selection_reason': (
                    'maximum_variance_in_component'
                    if feature == representative
                    else 'correlated_with_component_representative'
                ),
            })

    retained_set = set(retained)
    retained_in_original_order = [c for c in columns if c in retained_set]
    selection_table = pd.DataFrame(records).sort_values(
        ['cluster_id', 'status', 'feature'],
        ascending=[True, False, True],
    )
    return retained_in_original_order, selection_table, corr


def run_one_pca(X_raw, threshold, n_components=10):
    retained, selection_table, corr = select_descriptors_by_correlation(
        X_raw, threshold=threshold
    )
    X_selected = X_raw[retained]
    X_scaled = StandardScaler().fit_transform(X_selected)
    n_pc = min(n_components, X_scaled.shape[0], X_scaled.shape[1])
    if n_pc < 2:
        raise ValueError(
            f'Only {n_pc} PC can be computed at threshold {threshold}; '
            'at least two retained descriptors are required.'
        )
    pca = PCA(n_components=n_pc)
    scores = pca.fit_transform(X_scaled)
    return {
        'retained': retained,
        'selection_table': selection_table,
        'corr': corr,
        'scores': scores,
        'pca': pca,
    }


In [ ]:
# Run all datasets and thresholds.
results = {}
selection_tables = []
score_tables = []

for dataset, cfg in DATASETS.items():
    df = pd.read_csv(cfg['path'])
    id_column = df.columns[0]
    ids = clean_id_series(df[id_column], cfg['id_mode']).astype(str).tolist()
    feature_columns = [c for c in df.columns if str(c).endswith('_MolePct')]
    if not feature_columns:
        raise ValueError(f'No *_MolePct columns found in {cfg["path"]}')

    X_raw = df[feature_columns].astype(float).copy()
    X_raw.index = ids
    if X_raw.isna().any().any():
        bad = X_raw.columns[X_raw.isna().any()].tolist()
        raise ValueError(f'Missing values in {dataset}: {bad}')

    results[dataset] = {'ids': ids, 'X_raw': X_raw, 'runs': {}}

    for threshold in THRESHOLDS:
        run = run_one_pca(X_raw, threshold, n_components=N_COMPONENTS)
        results[dataset]['runs'][threshold] = run

        selection = run['selection_table'].copy()
        selection.insert(0, 'threshold', threshold)
        selection.insert(0, 'dataset', dataset)
        selection_tables.append(selection)

        score_df = pd.DataFrame(
            run['scores'],
            columns=[f'PC{i+1}' for i in range(run['scores'].shape[1])],
        )
        score_df.insert(0, 'ID', ids)
        score_df.insert(0, 'threshold', threshold)
        score_df.insert(0, 'dataset', dataset)
        score_tables.append(score_df)

        print(
            dataset,
            f'|r| > {threshold:.2f}:',
            len(run['retained']), 'retained;',
            f'PC1+PC2 = {run["pca"].explained_variance_ratio_[:2].sum()*100:.1f}%'
        )

selection_all = pd.concat(selection_tables, ignore_index=True)
scores_all = pd.concat(score_tables, ignore_index=True)
selection_all.to_csv(OUTPUT_DIR / 'descriptor_selection_all_thresholds.csv', index=False)
scores_all.to_csv(OUTPUT_DIR / 'pca_scores_all_thresholds.csv', index=False)


In [ ]:
# Quantify agreement with the primary threshold (0.60).
summary_rows = []

for dataset, dataset_result in results.items():
    reference = dataset_result['runs'][REFERENCE_THRESHOLD]['scores'][:, :2]
    reference_distances = pdist(reference, metric='euclidean')

    for threshold in THRESHOLDS:
        run = dataset_result['runs'][threshold]
        configuration = run['scores'][:, :2]
        distances = pdist(configuration, metric='euclidean')
        distance_rho, distance_p = spearmanr(reference_distances, distances)
        _, _, disparity = procrustes(reference, configuration)
        evr = run['pca'].explained_variance_ratio_

        summary_rows.append({
            'dataset': dataset,
            'threshold': threshold,
            'reference_threshold': REFERENCE_THRESHOLD,
            'n_input_descriptors': dataset_result['X_raw'].shape[1],
            'n_retained_descriptors': len(run['retained']),
            'n_excluded_descriptors': (
                dataset_result['X_raw'].shape[1] - len(run['retained'])
            ),
            'PC1_percent': evr[0] * 100,
            'PC2_percent': evr[1] * 100,
            'PC1_PC2_cumulative_percent': evr[:2].sum() * 100,
            'distance_spearman_rho_vs_r060': distance_rho,
            'distance_spearman_p_vs_r060': distance_p,
            'procrustes_disparity_vs_r060': disparity,
            'retained_descriptors': ';'.join(run['retained']),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'threshold_sensitivity_summary.csv', index=False)
display(summary.round(4))


In [ ]:
# Visual comparison of PC1-PC2 configurations.
fig, axes = plt.subplots(
    nrows=len(DATASETS),
    ncols=len(THRESHOLDS),
    figsize=(15, 11),
    constrained_layout=True,
)

for row, dataset in enumerate(DATASETS):
    for col, threshold in enumerate(THRESHOLDS):
        ax = axes[row, col]
        run = results[dataset]['runs'][threshold]
        scores = run['scores']
        evr = run['pca'].explained_variance_ratio_
        ax.scatter(scores[:, 0], scores[:, 1], s=13, alpha=0.75, color='#4c78a8')
        ax.axhline(0, color='#bdbdbd', linewidth=0.7)
        ax.axvline(0, color='#bdbdbd', linewidth=0.7)
        ax.set_title(
            f'{dataset}, |r|>{threshold:.2f}\n'
            f'n={len(run["retained"])}; PC1+PC2={evr[:2].sum()*100:.1f}%',
            fontsize=10,
            fontweight='bold' if threshold == REFERENCE_THRESHOLD else 'normal',
        )
        ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')

fig.savefig(OUTPUT_DIR / 'PC1_PC2_threshold_sensitivity.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'PC1_PC2_threshold_sensitivity.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Compact summary plots for the supplementary material.
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), constrained_layout=True)
colors = {'DBD': '#1f77b4', 'FL': '#d62728', 'non_DBD': '#2ca02c'}
metrics = [
    ('n_retained_descriptors', 'Retained descriptors'),
    ('PC1_PC2_cumulative_percent', 'PC1 + PC2 (%)'),
    ('distance_spearman_rho_vs_r060', 'Distance Spearman rho\nvs |r|>0.60'),
    ('procrustes_disparity_vs_r060', 'Procrustes disparity\nvs |r|>0.60'),
]

for ax, (metric, ylabel) in zip(axes, metrics):
    for dataset in DATASETS:
        part = summary[summary['dataset'] == dataset].sort_values('threshold')
        ax.plot(
            part['threshold'], part[metric], marker='o',
            label=dataset, color=colors[dataset]
        )
    ax.axvline(REFERENCE_THRESHOLD, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Correlation threshold')
    ax.set_ylabel(ylabel)
    ax.set_xticks(THRESHOLDS)

axes[0].legend(frameon=False)
fig.savefig(OUTPUT_DIR / 'threshold_sensitivity_metrics.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'threshold_sensitivity_metrics.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Create a README and download all outputs as one ZIP file.
readme = OUTPUT_DIR / 'README.txt'
readme.write_text(
    'PCA threshold sensitivity analysis\n'
    'Thresholds: |r| > 0.50, 0.60, 0.70, 0.80\n'
    'Reference threshold: |r| > 0.60\n'
    'Distance Spearman rho: higher values indicate more similar sample geometry.\n'
    'Procrustes disparity: lower values indicate more similar configurations.\n'
    'G1-G10 colors are not used here because this analysis evaluates PCA geometry itself.\n',
    encoding='utf-8',
)

zip_path = Path(shutil.make_archive(
    str(BASE_DIR / 'PCA_threshold_sensitivity_outputs'),
    'zip',
    root_dir=str(OUTPUT_DIR),
))
print('Created:', zip_path)

from google.colab import files
files.download(str(zip_path))
